In [2]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import Annotated
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# ===== Stateクラスの定義 =====
class State(TypedDict):
    messages: Annotated[list, add_messages]

# ===== グラフの構築 =====
def build_graph(model_name: str):
    # ツール定義（検索）
    tools = [TavilySearchResults(max_results=5)]

    # LLM
    llm = ChatOpenAI(model_name=model_name)
    llm_with_tools = llm.bind_tools(tools)

    graph_builder = StateGraph(State)

    # チャットボットノード
    def chatbot(state: State):
        return {"messages": [llm_with_tools.invoke(state["messages"])]}

    graph_builder.add_node("chatbot", chatbot)

    # ツールノード
    tool_node = ToolNode(tools)
    graph_builder.add_node("tools", tool_node)

    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )

    # tools → chatbot（ツール結果を踏まえて次の応答）
    graph_builder.add_edge("tools", "chatbot")

    # エントリポイント
    graph_builder.set_entry_point("chatbot")

    # メモリ付きでコンパイル
    memory = MemorySaver()
    graph = graph_builder.compile(checkpointer=memory)

    return graph

# ===== グラフ実行関数 =====
def stream_graph_updates(graph: StateGraph, user_input: str):
    events = graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values",
    )
    for event in events:
        message = event["messages"][-1]

        # assistant のメッセージだけ表示
        if message.type == "ai":
            print(message.content, flush=True)

# ===== メイン実行ロジック =====
# 環境変数の読み込み
load_dotenv("../.env")
os.environ["OPENAI_API_KEY"] = os.environ["API_KEY"]

# モデル名
MODEL_NAME = "gpt-4o-mini"

# グラフの作成
graph = build_graph(MODEL_NAME)

# メインループ
print("こんにちは！")
while True:
    user_input = input("質問:")
    if user_input.strip() == "":
        print("ありがとうございました!")
        break
    stream_graph_updates(graph, user_input)

こんにちは！

今日の天気に関する情報は以下の通りです：

- **全国的な天気概況**: 西～東日本では雲が広がりやすく、所々でにわか雨がある見込みですが、西日本は午後から天気が回復する可能性があります。北日本は概ね晴れていますが、東北北部では雨や雪の降る所がありそうです。沖縄では曇りや雨が予想されています。

- **主要都市の天気**:
  - **東京**: 曇り時々晴れ
  - **大阪**: 曇りのち晴れ
  - **福岡**: 曇り時々晴れ
  - **仙台**: 曇り時々晴れ

- **気温**: 全国的に平年並みか高めの気温が予想されています。

詳細な情報や最新の天気予報は、[こちらのリンク](https://weather.yahoo.co.jp/weather/)からご覧いただけます。
ありがとうございました!
